# ADL Homework 4 — Convolutional Neural Networks on CIFAR-10

**Applied Deep Learning, Spring 2026**

We implement the PyTorch CIFAR-10 tutorial CNN, train it for image classification, and (in later tasks) extend it with a deconvolutional decoder for reconstruction and latent-feature analysis.

## Hyperparameters (Task 1)

We follow the [PyTorch CIFAR-10 tutorial](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) and the Lecture 7 demo. Because computational resources are limited, we train on **6,000 images** (one tenth of the CIFAR-10 training set), as allowed by the assignment.

| Parameter | Value |
|-----------|-------|
| Architecture | Conv(3→6, k=5) → ReLU → MaxPool → Conv(6→16, k=5) → ReLU → MaxPool → FC(400→120) → ReLU → FC(120→84) → ReLU → FC(84→10) |
| Loss | Cross-entropy |
| Optimizer | SGD (lr = 0.001, momentum = 0.9) |
| Batch size | 64 |
| Epochs | 40 |
| Training subset | **6,000** images (random sample, seed 42) |
| Test set | 10,000 images (full CIFAR-10 test set) |
| Input normalization | `ToTensor` + normalize to [-1, 1] with mean/std (0.5, 0.5, 0.5) |

In [ ]:
%matplotlib inline

from typing import List, cast

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.axes import Axes
from matplotlib.figure import Figure

from cifar_cnn import (
    CIFAR10_CLASSES,
    DEFAULT_BATCH_SIZE,
    DEFAULT_EPOCHS,
    DEFAULT_TRAIN_SUBSET,
    Net,
    collect_predictions,
    denormalize,
    get_device,
    make_dataloaders,
    per_class_accuracy,
    set_seed,
    train_model,
)

plt.rcParams.update({"figure.figsize": (8, 5), "font.size": 11})

SEED = 42
TRAIN_SUBSET = DEFAULT_TRAIN_SUBSET
EPOCHS = DEFAULT_EPOCHS
BATCH_SIZE = DEFAULT_BATCH_SIZE

set_seed(SEED)
device = get_device()
print(f"Device: {device}")
print(f"Training on {TRAIN_SUBSET} images for {EPOCHS} epochs")

## Task 1 — CIFAR-10 classification

We train the CNN below and report training/test accuracy, learning curves, sample test predictions, and per-class accuracy.

In [ ]:
trainloader, testloader = make_dataloaders(
    train_subset_size=TRAIN_SUBSET,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

model = Net().to(device)
history = train_model(
    model,
    trainloader,
    testloader,
    device,
    epochs=EPOCHS,
)
per_class = per_class_accuracy(model, testloader, device)

final_train_acc = 100 * history[-1].train_acc
final_test_acc = 100 * history[-1].test_acc
print(f"\nFinal train accuracy: {final_train_acc:.1f}%")
print(f"Final test accuracy:  {final_test_acc:.1f}%")

In [ ]:
print("Epoch | train loss | train acc (%) | test loss | test acc (%)")
print("-" * 58)
for row in history:
    print(
        f"{row.epoch:5d} | {row.train_loss:10.3f} | "
        f"{100 * row.train_acc:13.1f} | {row.test_loss:9.3f} | "
        f"{100 * row.test_acc:12.1f}"
    )

print("\nPer-class test accuracy:")
print("Class   | Accuracy (%)")
print("-" * 22)
for cls in CIFAR10_CLASSES:
    print(f"{cls:7s} | {per_class[cls]:11.1f}")

In [ ]:
epochs = [row.epoch for row in history]
train_acc = [100 * row.train_acc for row in history]
test_acc = [100 * row.test_acc for row in history]

fig, ax = plt.subplots(figsize=(7, 4))
figure = cast(Figure, fig)
axis = cast(Axes, ax)
axis.plot(epochs, train_acc, marker="o", label="Train")
axis.plot(epochs, test_acc, marker="o", label="Test")
axis.set_xlabel("Epoch")
axis.set_ylabel("Accuracy (%)")
axis.set_title("Task 1: CIFAR-10 classification accuracy")
axis.legend()
axis.grid(True, alpha=0.3)
figure.tight_layout()
plt.show()

In [ ]:
images, labels, preds = collect_predictions(model, testloader, device, max_images=10)

n = images.size(0)
ncols = 5
nrows = (n + ncols - 1) // ncols

fig, axes_raw = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.2 * nrows))
figure = cast(Figure, fig)
axes = cast(List[Axes], list(np.atleast_1d(axes_raw).ravel()))

for idx in range(n):
    img = denormalize(images[idx]).numpy().transpose(1, 2, 0)
    true_name = CIFAR10_CLASSES[labels[idx]]
    pred_name = CIFAR10_CLASSES[preds[idx]]
    color = "green" if labels[idx] == preds[idx] else "red"
    axes[idx].imshow(np.clip(img, 0, 1))
    axes[idx].set_title(f"true: {true_name}\npred: {pred_name}", color=color, fontsize=8)
    axes[idx].axis("off")

for idx in range(n, len(axes)):
    axes[idx].axis("off")

figure.suptitle("Test images with predicted labels", fontsize=11)
figure.tight_layout()
plt.show()

### Discussion

On the reduced training set (6,000 images) and limited compute, the network typically reaches mid-40s percent test accuracy after 40 epochs. This is below the full-tutorial result on 50,000 images (~55% in 2 epochs), which is expected with less data and fewer total optimization steps. Vehicles (`car`, `truck`, `plane`) and `dog` are often classified more reliably than fine-grained categories such as `cat` and `bird`. Methodology matches the tutorial; accuracy is limited mainly by the smaller training subset and compute budget.